# Last.fm Global Trends — Data Fetcher

Fetches top artists and tracks for all countries (plus global charts) from the Last.fm API and saves them to a DuckDB database.

**Before running:**
1. Enable **Internet** in Notebook Settings (right panel)
2. Add your Last.fm credentials as Kaggle Secrets:
   - `LASTFM_API_KEY`
   - `LASTFM_API_SECRET`
3. Adjust the configuration cell below if needed

In [ ]:
!pip install -q pylast duckdb pandas

## Configuration

| Variable | Description |
|---|---|
| `GLOBAL_MAX_AGE_HOURS` | Re-fetch global charts (artists, tracks, tags) older than this. Default: 72h (3 days). Set to `0` to force a full refresh. |
| `GEO_MAX_AGE_HOURS` | Re-fetch country charts older than this. Default: 168h (7 days). Set to `0` to force a full refresh. |
| `ONLY_COUNTRIES` | Limit to specific countries. Empty list = all countries. |
| `DB_PATH` | Where to save the DuckDB file. |

In [ ]:
GLOBAL_MAX_AGE_HOURS = 72    # 3 days. Set to 0 to force full refresh of global charts.
GEO_MAX_AGE_HOURS = 168      # 7 days. Set to 0 to force full refresh of country charts.
ONLY_COUNTRIES = []          # e.g. ["France", "Germany"]. Empty = all countries.
DB_PATH = "/kaggle/working/trends.db"

In [ ]:
import os
import shutil

# Remove stale WAL file left by a crashed/interrupted previous session.
# Without this, DuckDB throws a CatalogException when replaying the WAL
# against a DB that already has the tables created.
_WAL_PATH = f"{DB_PATH}.wal"
if os.path.exists(_WAL_PATH):
    os.remove(_WAL_PATH)
    print(f"Removed stale WAL file: {_WAL_PATH}")

# Seed the working DB from the Kaggle input dataset.
# - If working DB doesn't exist → copy from input.
# - If working DB exists but differs in size → input has newer data, overwrite.
# - If working DB exists and same size → already up to date, skip.
_INPUT_DB = "/kaggle/input/datasets/tiagoadrianunes/last-fm-global-trends/trends.db"

if not os.path.exists(_INPUT_DB):
    print(f"No input DB found at {_INPUT_DB} — will start fresh.")
elif not os.path.exists(DB_PATH):
    shutil.copy2(_INPUT_DB, DB_PATH)
    print(f"Copied input DB to {DB_PATH} ({os.path.getsize(DB_PATH):,} bytes)")
elif os.path.getsize(_INPUT_DB) != os.path.getsize(DB_PATH):
    input_size = os.path.getsize(_INPUT_DB)
    working_size = os.path.getsize(DB_PATH)
    shutil.copy2(_INPUT_DB, DB_PATH)
    print(f"Input DB differs in size (input={input_size:,}  working={working_size:,}) — overwritten.")
else:
    print(f"Working DB matches input size ({os.path.getsize(DB_PATH):,} bytes) — skipping copy.")

## Credentials

Reads `LASTFM_API_KEY` and `LASTFM_API_SECRET` from Kaggle Secrets.
If running locally, set them as environment variables instead.

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    os.environ["LASTFM_API_KEY"] = _secrets.get_secret("LASTFM_API_KEY")
    os.environ["LASTFM_API_SECRET"] = _secrets.get_secret("LASTFM_API_SECRET")
    print("Credentials loaded from Kaggle Secrets.")
except Exception:
    print("Kaggle Secrets not available — using environment variables.")

assert os.environ.get("LASTFM_API_KEY"), "LASTFM_API_KEY is not set"
assert os.environ.get("LASTFM_API_SECRET"), "LASTFM_API_SECRET is not set"

## Imports & Constants

In [ ]:
import contextlib
import json
import logging
import time
from datetime import datetime
from pathlib import Path

import duckdb
import pandas as pd
import pylast

FETCH_LIMIT    = 1000  # results per page (API max)
TAGS_MAX_PAGES = 10    # hard cap for chart.getTopTags (10 × 1 000 = 10 000 tags)
REQUEST_DELAY  = 1     # seconds between API calls
MAX_RETRIES    = 3     # retries for transient errors
RETRY_BACKOFF  = 5     # seconds between retries

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s",
)
log = logging.getLogger(__name__)

## Country Maps

In [ ]:
# ISO 3166-1 country names mapped to alpha-2 codes
COUNTRY_CODES: dict[str, str] = {
    "Afghanistan": "AF", "Åland Islands": "AX", "Albania": "AL", "Algeria": "DZ",
    "American Samoa": "AS", "Andorra": "AD", "Angola": "AO", "Anguilla": "AI",
    "Antarctica": "AQ", "Antigua and Barbuda": "AG", "Argentina": "AR", "Armenia": "AM",
    "Aruba": "AW", "Australia": "AU", "Austria": "AT", "Azerbaijan": "AZ",
    "Bahamas": "BS", "Bahrain": "BH", "Bangladesh": "BD", "Barbados": "BB",
    "Belarus": "BY", "Belgium": "BE", "Belize": "BZ", "Benin": "BJ",
    "Bermuda": "BM", "Bhutan": "BT", "Bolivia": "BO",
    "Bonaire, Sint Eustatius and Saba": "BQ",
    "Bosnia and Herzegovina": "BA", "Botswana": "BW", "Bouvet Island": "BV",
    "Brazil": "BR", "British Indian Ocean Territory": "IO", "Brunei Darussalam": "BN",
    "Bulgaria": "BG", "Burkina Faso": "BF", "Burundi": "BI", "Cabo Verde": "CV",
    "Cambodia": "KH", "Cameroon": "CM", "Canada": "CA", "Cayman Islands": "KY",
    "Central African Republic": "CF", "Chad": "TD", "Chile": "CL", "China": "CN",
    "Christmas Island": "CX", "Cocos (Keeling) Islands": "CC", "Colombia": "CO",
    "Comoros": "KM", "Congo, The Democratic Republic Of The": "CD", "Congo": "CG",
    "Cook Islands": "CK", "Costa Rica": "CR", "Côte d'Ivoire": "CI", "Croatia": "HR",
    "Cuba": "CU", "Curaçao": "CW", "Cyprus": "CY", "Czechia": "CZ",
    "Denmark": "DK", "Djibouti": "DJ", "Dominica": "DM", "Dominican Republic": "DO",
    "Ecuador": "EC", "Egypt": "EG", "El Salvador": "SV", "Equatorial Guinea": "GQ",
    "Eritrea": "ER", "Estonia": "EE", "Eswatini": "SZ", "Ethiopia": "ET",
    "Falkland Islands (Malvinas)": "FK", "Faroe Islands": "FO", "Fiji": "FJ",
    "Finland": "FI", "France": "FR", "French Guiana": "GF", "French Polynesia": "PF",
    "French Southern Territories": "TF", "Gabon": "GA", "Gambia": "GM",
    "Georgia": "GE", "Germany": "DE", "Ghana": "GH", "Gibraltar": "GI",
    "Greece": "GR", "Greenland": "GL", "Grenada": "GD", "Guadeloupe": "GP",
    "Guam": "GU", "Guatemala": "GT", "Guernsey": "GG", "Guinea": "GN",
    "Guinea-Bissau": "GW", "Guyana": "GY", "Haiti": "HT",
    "Heard Island and McDonald Islands": "HM", "Holy See (Vatican City State)": "VA",
    "Honduras": "HN", "Hong Kong": "HK", "Hungary": "HU", "Iceland": "IS",
    "India": "IN", "Indonesia": "ID", "Iran, Islamic Republic of": "IR", "Iraq": "IQ",
    "Ireland": "IE", "Isle of Man": "IM", "Israel": "IL", "Italy": "IT",
    "Jamaica": "JM", "Japan": "JP", "Jersey": "JE", "Jordan": "JO",
    "Kazakhstan": "KZ", "Kenya": "KE", "Kiribati": "KI",
    "Korea, Democratic People's Republic of": "KP", "Korea, Republic of": "KR",
    "Kuwait": "KW", "Kyrgyzstan": "KG", "Lao People's Democratic Republic": "LA",
    "Latvia": "LV", "Lebanon": "LB", "Lesotho": "LS", "Liberia": "LR",
    "Libya": "LY", "Liechtenstein": "LI", "Lithuania": "LT", "Luxembourg": "LU",
    "Macao": "MO", "Madagascar": "MG", "Malawi": "MW", "Malaysia": "MY",
    "Maldives": "MV", "Mali": "ML", "Malta": "MT", "Marshall Islands": "MH",
    "Martinique": "MQ", "Mauritania": "MR", "Mauritius": "MU", "Mayotte": "YT",
    "Mexico": "MX", "Micronesia, Federated States of": "FM", "Moldova": "MD",
    "Monaco": "MC", "Mongolia": "MN", "Montenegro": "ME", "Montserrat": "MS",
    "Morocco": "MA", "Mozambique": "MZ", "Myanmar": "MM", "Namibia": "NA",
    "Nauru": "NR", "Nepal": "NP", "Netherlands": "NL", "New Caledonia": "NC",
    "New Zealand": "NZ", "Nicaragua": "NI", "Niger": "NE", "Nigeria": "NG",
    "Niue": "NU", "Norfolk Island": "NF", "North Macedonia": "MK",
    "Northern Mariana Islands": "MP", "Norway": "NO", "Oman": "OM",
    "Pakistan": "PK", "Palestine": "PS", "Palau": "PW", "Panama": "PA",
    "Papua New Guinea": "PG", "Paraguay": "PY", "Peru": "PE", "Philippines": "PH",
    "Pitcairn": "PN", "Poland": "PL", "Portugal": "PT", "Puerto Rico": "PR",
    "Qatar": "QA", "Réunion": "RE", "Romania": "RO", "Russian Federation": "RU",
    "Rwanda": "RW", "Saint Barthélemy": "BL", "Saint Helena": "SH",
    "Saint Kitts and Nevis": "KN", "Saint Lucia": "LC",
    "Saint Martin (French part)": "MF", "Saint Pierre and Miquelon": "PM",
    "Saint Vincent and the Grenadines": "VC", "Samoa": "WS", "San Marino": "SM",
    "Sao Tome and Principe": "ST", "Saudi Arabia": "SA", "Senegal": "SN",
    "Serbia": "RS", "Seychelles": "SC", "Sierra Leone": "SL", "Singapore": "SG",
    "Sint Maarten (Dutch part)": "SX", "Slovakia": "SK", "Slovenia": "SI",
    "Solomon Islands": "SB", "Somalia": "SO", "South Africa": "ZA",
    "South Georgia and the South Sandwich Islands": "GS", "South Sudan": "SS",
    "Spain": "ES", "Sri Lanka": "LK", "Sudan": "SD", "Suriname": "SR",
    "Svalbard and Jan Mayen": "SJ", "Sweden": "SE", "Switzerland": "CH",
    "Syrian Arab Republic": "SY", "Taiwan": "TW", "Tajikistan": "TJ",
    "Tanzania, United Republic of": "TZ", "Thailand": "TH", "Timor-Leste": "TL",
    "Togo": "TG", "Tokelau": "TK", "Tonga": "TO", "Trinidad and Tobago": "TT",
    "Tunisia": "TN", "Türkiye": "TR", "Turkmenistan": "TM",
    "Turks and Caicos Islands": "TC", "Tuvalu": "TV", "Uganda": "UG",
    "Ukraine": "UA", "United Arab Emirates": "AE", "United Kingdom": "GB",
    "United States Minor Outlying Islands": "UM", "United States": "US",
    "Uruguay": "UY", "Uzbekistan": "UZ", "Vanuatu": "VU", "Venezuela": "VE",
    "Viet Nam": "VN", "Virgin Islands, British": "VG", "Virgin Islands, U.S.": "VI",
    "Wallis and Futuna": "WF", "Western Sahara": "EH", "Yemen": "YE",
    "Zambia": "ZM", "Zimbabwe": "ZW",
}

# Maps display names to the name strings accepted by the Last.fm geo API.
# None = not supported by Last.fm.
LASTFM_COUNTRY_NAME_MAP: dict[str, str | None] = {
    "Côte d'Ivoire": "Cote D'Ivoire",
    "Cabo Verde": "Cape Verde",
    "Czechia": "Czech Republic",
    "Eswatini": "Swaziland",
    "Libya": "Libyan Arab Jamahiriya",  # renamed ~2011; Last.fm hasn't updated
    "North Macedonia": "Macedonia",
    "Réunion": "Reunion",
    "Türkiye": "Turkey",
    "Bonaire, Sint Eustatius and Saba": None,
    "Palestine": None,
    "Sint Maarten (Dutch part)": None,
}

ALL_COUNTRIES = list(COUNTRY_CODES.keys())
print(f"{len(ALL_COUNTRIES)} countries loaded")

## Database Setup

In [ ]:
def text(node, tag: str) -> str:
    """Extract text content of the first matching child XML element."""
    els = node.getElementsByTagName(tag)
    if els and els[0].firstChild:
        return els[0].firstChild.nodeValue or ""
    return ""


def setup_db(con: duckdb.DuckDBPyConnection) -> set[str]:
    """Create tables and migrate schema. Returns set of table names that had new columns added."""
    con.execute("""
        CREATE TABLE IF NOT EXISTS geo_top_artists (
            country     VARCHAR,
            rank        INTEGER,
            artist      VARCHAR,
            artist_mbid VARCHAR,
            artist_url  VARCHAR,
            listeners   INTEGER,
            fetched_at  TIMESTAMP DEFAULT current_timestamp,
            PRIMARY KEY (country, rank)
        )
    """)
    con.execute("""
        CREATE TABLE IF NOT EXISTS geo_top_tracks (
            country     VARCHAR,
            rank        INTEGER,
            track       VARCHAR,
            track_url   VARCHAR,
            artist      VARCHAR,
            artist_mbid VARCHAR,
            artist_url  VARCHAR,
            listeners   INTEGER,
            playcount   INTEGER,
            fetched_at  TIMESTAMP DEFAULT current_timestamp,
            PRIMARY KEY (country, rank)
        )
    """)
    con.execute("""
        CREATE TABLE IF NOT EXISTS global_top_artists (
            rank        INTEGER PRIMARY KEY,
            artist      VARCHAR,
            artist_mbid VARCHAR,
            artist_url  VARCHAR,
            listeners   BIGINT,
            playcount   BIGINT,
            fetched_at  TIMESTAMP DEFAULT current_timestamp
        )
    """)
    con.execute("""
        CREATE TABLE IF NOT EXISTS global_top_tracks (
            rank        INTEGER PRIMARY KEY,
            track       VARCHAR,
            track_mbid  VARCHAR,
            track_url   VARCHAR,
            artist      VARCHAR,
            artist_mbid VARCHAR,
            artist_url  VARCHAR,
            listeners   BIGINT,
            playcount   BIGINT,
            fetched_at  TIMESTAMP DEFAULT current_timestamp
        )
    """)
    con.execute("""
        CREATE TABLE IF NOT EXISTS global_top_tags (
            rank       INTEGER PRIMARY KEY,
            tag        VARCHAR,
            tag_url    VARCHAR,
            reach      INTEGER,
            taggings   INTEGER,
            fetched_at TIMESTAMP DEFAULT current_timestamp
        )
    """)
    # Migrate existing DBs that may be missing new columns.
    # Track which tables gain a new column — those need a forced refresh.
    schema_changed: set[str] = set()
    for tbl, col, typ in [
        ("geo_top_artists",   "artist_mbid", "VARCHAR"),
        ("geo_top_artists",   "artist_url",  "VARCHAR"),
        ("geo_top_tracks",    "track_url",   "VARCHAR"),
        ("geo_top_tracks",    "artist_mbid", "VARCHAR"),
        ("geo_top_tracks",    "artist_url",  "VARCHAR"),
        ("geo_top_tracks",    "playcount",   "INTEGER"),
        ("global_top_artists","artist_mbid", "VARCHAR"),
        ("global_top_artists","artist_url",  "VARCHAR"),
        ("global_top_artists","listeners",   "BIGINT"),
        ("global_top_artists","playcount",   "BIGINT"),
        ("global_top_tracks", "track_mbid",  "VARCHAR"),
        ("global_top_tracks", "track_url",   "VARCHAR"),
        ("global_top_tracks", "artist_mbid", "VARCHAR"),
        ("global_top_tracks", "artist_url",  "VARCHAR"),
        ("global_top_tracks", "listeners",   "BIGINT"),
        ("global_top_tracks", "playcount",   "BIGINT"),
    ]:
        try:
            con.execute(f"ALTER TABLE {tbl} ADD COLUMN {col} {typ}")
            schema_changed.add(tbl)
            log.info("Schema migration: added %s.%s — forcing full refresh for %s", tbl, col, tbl)
        except duckdb.CatalogException:
            pass
        with contextlib.suppress(duckdb.CatalogException):
            con.execute(f"ALTER TABLE {tbl} ALTER COLUMN {col} TYPE {typ}")

    print("Database ready:", DB_PATH)
    return schema_changed

## Helper Functions

In [ ]:
def _last_fetched(con, table: str, country: str | None) -> datetime | None:
    if country is None:
        result = con.execute(f"SELECT MAX(fetched_at) FROM {table}").fetchone()
    else:
        result = con.execute(
            f"SELECT MAX(fetched_at) FROM {table} WHERE country = ?", [country]
        ).fetchone()
    return result[0] if result and result[0] is not None else None


def _is_stale(con, table: str, country: str | None, max_age_hours: float) -> bool:
    where = "WHERE country = ?" if country is not None else ""
    params = [max_age_hours] + ([country] if country is not None else [])
    row = con.execute(
        f"SELECT MAX(fetched_at) IS NULL OR "
        f"(epoch(now()) - epoch(MAX(fetched_at))) / 3600.0 > ? "
        f"FROM {table} {where}",
        params,
    ).fetchone()
    return bool(row[0])


def _db_count(con, table: str, country: str | None) -> int:
    if country is None:
        return con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    return con.execute(
        f"SELECT COUNT(*) FROM {table} WHERE country = ?", [country]
    ).fetchone()[0]


def _is_unchanged(con, table: str, country: str, rows: list[dict]) -> bool:
    result = con.execute(
        f"SELECT COUNT(*), COALESCE(SUM(listeners), 0) FROM {table} WHERE country = ?",
        [country],
    ).fetchone()
    db_count, db_sum = result
    return db_count == len(rows) and db_sum == sum(r["listeners"] for r in rows)


def _total_pages(doc, wrapper_tag: str) -> int:
    nodes = doc.getElementsByTagName(wrapper_tag)
    if nodes and "totalPages" in nodes[0].attributes:
        return int(nodes[0].attributes["totalPages"].value)
    return 1


def _parse_page1_artists(doc, country: str) -> list[dict]:
    return [
        {
            "country": country,
            "rank": i,
            "artist": text(el, "name"),
            "artist_mbid": text(el, "mbid"),
            "artist_url": text(el, "url"),
            "listeners": int(text(el, "listeners") or 0),
        }
        for i, el in enumerate(doc.getElementsByTagName("artist"), start=1)
    ]


def _parse_page1_tracks(doc, country: str) -> list[dict]:
    rows = []
    for i, el in enumerate(doc.getElementsByTagName("track"), start=1):
        artist_nodes = el.getElementsByTagName("artist")
        rows.append({
            "country": country,
            "rank": i,
            "track": text(el, "name"),
            "track_url": text(el, "url"),
            "artist": text(artist_nodes[0], "name") if artist_nodes else "",
            "artist_mbid": text(artist_nodes[0], "mbid") if artist_nodes else "",
            "artist_url": text(artist_nodes[0], "url") if artist_nodes else "",
            "listeners": int(text(el, "listeners") or 0),
            "playcount": int(text(el, "playcount") or 0),
        })
    return rows


def _raw_request(network, method: str, country: str, run_errors: list, page: int = 1):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            req = pylast._Request(
                network, method,
                {"country": country, "limit": FETCH_LIMIT, "page": page},
            )
            return req.execute(cacheable=False)
        except pylast.WSError as e:
            msg = f"{method} for {country} (page {page}): {e}"
            log.warning(msg)
            run_errors.append(msg)
            return None
        except (pylast.MalformedResponseError, pylast.NetworkError) as e:
            if attempt < MAX_RETRIES:
                log.warning("%s for %s (page %d) attempt %d/%d failed: %s — retrying in %ds",
                            method, country, page, attempt, MAX_RETRIES, e, RETRY_BACKOFF)
                time.sleep(RETRY_BACKOFF)
            else:
                msg = f"{method} for {country} (page {page}) failed after {MAX_RETRIES} attempts: {e}"
                log.error(msg)
                run_errors.append(msg)
                return None


def _global_request(network, method: str, run_errors: list, page: int = 1):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            req = pylast._Request(network, method, {"limit": FETCH_LIMIT, "page": page})
            return req.execute(cacheable=False)
        except pylast.WSError as e:
            msg = f"{method} (page {page}): {e}"
            log.warning(msg)
            run_errors.append(msg)
            return None
        except (pylast.MalformedResponseError, pylast.NetworkError) as e:
            if attempt < MAX_RETRIES:
                log.warning("%s (page %d) attempt %d/%d failed: %s — retrying in %ds",
                            method, page, attempt, MAX_RETRIES, e, RETRY_BACKOFF)
                time.sleep(RETRY_BACKOFF)
            else:
                msg = f"{method} (page {page}) failed after {MAX_RETRIES} attempts: {e}"
                log.error(msg)
                run_errors.append(msg)
                return None

## Fetch Functions

In [ ]:
def fetch_geo_artists(network, country: str, con, run_errors: list, max_age_hours: float):
    stale = _is_stale(con, "geo_top_artists", country, max_age_hours)
    log.info("Fetching geo.getTopArtists — %s%s", country, " (stale)" if stale else "")
    doc = _raw_request(network, "geo.getTopArtists", country, run_errors, page=1)
    if doc is None:
        return [], False
    total_pages = _total_pages(doc, "topartists")
    db_count = _db_count(con, "geo_top_artists", country)
    count_ok = (total_pages - 1) * FETCH_LIMIT < db_count <= total_pages * FETCH_LIMIT
    if not stale and count_ok:
        log.info("  artists up to date (%d rows), skipping", db_count)
        return [], False
    if not stale and not count_ok:
        log.info("  count mismatch (db=%d expected %d–%d) — re-fetching despite fresh data",
                 db_count, (total_pages - 1) * FETCH_LIMIT + 1, total_pages * FETCH_LIMIT)
    rows = _parse_page1_artists(doc, country)
    for page in range(2, total_pages + 1):
        time.sleep(REQUEST_DELAY)
        doc = _raw_request(network, "geo.getTopArtists", country, run_errors, page=page)
        if doc is None:
            break
        offset = (page - 1) * FETCH_LIMIT
        for i, el in enumerate(doc.getElementsByTagName("artist"), start=1):
            rows.append({
                "country": country,
                "rank": offset + i,
                "artist": text(el, "name"),
                "artist_mbid": text(el, "mbid"),
                "artist_url": text(el, "url"),
                "listeners": int(text(el, "listeners") or 0),
            })
    log.info("  → %d rows (%d pages)", len(rows), total_pages)
    return rows, True


def fetch_geo_tracks(network, country: str, con, run_errors: list, max_age_hours: float):
    stale = _is_stale(con, "geo_top_tracks", country, max_age_hours)
    log.info("Fetching geo.getTopTracks — %s%s", country, " (stale)" if stale else "")
    doc = _raw_request(network, "geo.getTopTracks", country, run_errors, page=1)
    if doc is None:
        return [], False
    total_pages = _total_pages(doc, "tracks")
    db_count = _db_count(con, "geo_top_tracks", country)
    count_ok = (total_pages - 1) * FETCH_LIMIT < db_count <= total_pages * FETCH_LIMIT
    if not stale and count_ok:
        log.info("  tracks up to date (%d rows), skipping", db_count)
        return [], False
    if not stale and not count_ok:
        log.info("  count mismatch (db=%d expected %d–%d) — re-fetching despite fresh data",
                 db_count, (total_pages - 1) * FETCH_LIMIT + 1, total_pages * FETCH_LIMIT)
    rows = _parse_page1_tracks(doc, country)
    for page in range(2, total_pages + 1):
        time.sleep(REQUEST_DELAY)
        doc = _raw_request(network, "geo.getTopTracks", country, run_errors, page=page)
        if doc is None:
            break
        offset = (page - 1) * FETCH_LIMIT
        for i, el in enumerate(doc.getElementsByTagName("track"), start=1):
            artist_nodes = el.getElementsByTagName("artist")
            rows.append({
                "country": country,
                "rank": offset + i,
                "track": text(el, "name"),
                "track_url": text(el, "url"),
                "artist": text(artist_nodes[0], "name") if artist_nodes else "",
                "artist_mbid": text(artist_nodes[0], "mbid") if artist_nodes else "",
                "artist_url": text(artist_nodes[0], "url") if artist_nodes else "",
                "listeners": int(text(el, "listeners") or 0),
                "playcount": int(text(el, "playcount") or 0),
            })
    log.info("  → %d rows (%d pages)", len(rows), total_pages)
    return rows, True


def fetch_global_artists(network, con, run_errors: list, max_age_hours: float):
    stale = _is_stale(con, "global_top_artists", None, max_age_hours)
    log.info("Fetching chart.getTopArtists — global%s", " (stale)" if stale else "")
    doc = _global_request(network, "chart.getTopArtists", run_errors, page=1)
    if doc is None:
        return []
    total_pages = _total_pages(doc, "artists")
    db_count = _db_count(con, "global_top_artists", None)
    count_ok = (total_pages - 1) * FETCH_LIMIT < db_count <= total_pages * FETCH_LIMIT
    if not stale and count_ok:
        log.info("  global artists up to date (%d rows), skipping", db_count)
        return []
    if not stale and not count_ok:
        log.info("  count mismatch (db=%d expected %d–%d) — re-fetching despite fresh data",
                 db_count, (total_pages - 1) * FETCH_LIMIT + 1, total_pages * FETCH_LIMIT)
    rows = []
    for page in range(1, total_pages + 1):
        if page > 1:
            time.sleep(REQUEST_DELAY)
            doc = _global_request(network, "chart.getTopArtists", run_errors, page=page)
            if doc is None:
                break
        offset = (page - 1) * FETCH_LIMIT
        for i, el in enumerate(doc.getElementsByTagName("artist"), start=1):
            rows.append({
                "rank": offset + i,
                "artist": text(el, "name"),
                "artist_mbid": text(el, "mbid"),
                "artist_url": text(el, "url"),
                "listeners": int(text(el, "listeners") or 0),
                "playcount": int(text(el, "playcount") or 0),
            })
    log.info("  → %d rows (%d pages)", len(rows), total_pages)
    return rows


def fetch_global_tracks(network, con, run_errors: list, max_age_hours: float):
    stale = _is_stale(con, "global_top_tracks", None, max_age_hours)
    log.info("Fetching chart.getTopTracks — global%s", " (stale)" if stale else "")
    doc = _global_request(network, "chart.getTopTracks", run_errors, page=1)
    if doc is None:
        return []
    total_pages = _total_pages(doc, "tracks")
    db_count = _db_count(con, "global_top_tracks", None)
    count_ok = (total_pages - 1) * FETCH_LIMIT < db_count <= total_pages * FETCH_LIMIT
    if not stale and count_ok:
        log.info("  global tracks up to date (%d rows), skipping", db_count)
        return []
    if not stale and not count_ok:
        log.info("  count mismatch (db=%d expected %d–%d) — re-fetching despite fresh data",
                 db_count, (total_pages - 1) * FETCH_LIMIT + 1, total_pages * FETCH_LIMIT)
    rows = []
    for page in range(1, total_pages + 1):
        if page > 1:
            time.sleep(REQUEST_DELAY)
            doc = _global_request(network, "chart.getTopTracks", run_errors, page=page)
            if doc is None:
                break
        offset = (page - 1) * FETCH_LIMIT
        for i, el in enumerate(doc.getElementsByTagName("track"), start=1):
            artist_nodes = el.getElementsByTagName("artist")
            rows.append({
                "rank": offset + i,
                "track": text(el, "name"),
                "track_mbid": text(el, "mbid"),
                "track_url": text(el, "url"),
                "artist": text(artist_nodes[0], "name") if artist_nodes else "",
                "artist_mbid": text(artist_nodes[0], "mbid") if artist_nodes else "",
                "artist_url": text(artist_nodes[0], "url") if artist_nodes else "",
                "listeners": int(text(el, "listeners") or 0),
                "playcount": int(text(el, "playcount") or 0),
            })
    log.info("  → %d rows (%d pages)", len(rows), total_pages)
    return rows


def fetch_global_tags(network, con, run_errors: list, max_age_hours: float):
    stale = _is_stale(con, "global_top_tags", None, max_age_hours)
    log.info("Fetching chart.getTopTags — global%s (capped at %d pages)",
             " (stale)" if stale else "", TAGS_MAX_PAGES)
    doc = _global_request(network, "chart.getTopTags", run_errors, page=1)
    if doc is None:
        return []
    total_pages = min(_total_pages(doc, "tags"), TAGS_MAX_PAGES)
    db_count = _db_count(con, "global_top_tags", None)
    count_ok = (total_pages - 1) * FETCH_LIMIT < db_count <= total_pages * FETCH_LIMIT
    if not stale and count_ok:
        log.info("  global tags up to date (%d rows), skipping", db_count)
        return []
    if not stale and not count_ok:
        log.info("  count mismatch (db=%d expected %d–%d) — re-fetching despite fresh data",
                 db_count, (total_pages - 1) * FETCH_LIMIT + 1, total_pages * FETCH_LIMIT)
    rows = []
    for page in range(1, total_pages + 1):
        if page > 1:
            time.sleep(REQUEST_DELAY)
            doc = _global_request(network, "chart.getTopTags", run_errors, page=page)
            if doc is None:
                break
        offset = (page - 1) * FETCH_LIMIT
        for i, el in enumerate(doc.getElementsByTagName("tag"), start=1):
            rows.append({
                "rank": offset + i,
                "tag": text(el, "name"),
                "tag_url": text(el, "url"),
                "reach": int(text(el, "reach") or 0),
                "taggings": int(text(el, "taggings") or 0),
            })
    log.info("  → %d rows (%d pages, capped at %d)", len(rows), total_pages, TAGS_MAX_PAGES)
    return rows

## Upsert Functions

In [ ]:
def upsert_artists(con, rows: list[dict], force: bool, run_errors: list) -> bool:
    if not rows:
        return True
    if not force and _is_unchanged(con, "geo_top_artists", rows[0]["country"], rows):
        log.info("  artists unchanged, skipping")
        return True
    country = rows[0]["country"]
    df = pd.DataFrame(rows)
    con.register("_staging_artists", df)
    con.execute("DELETE FROM geo_top_artists WHERE country = ?", [country])
    con.execute("""
        INSERT INTO geo_top_artists (country, rank, artist, artist_mbid, artist_url, listeners)
        SELECT country, rank, artist, artist_mbid, artist_url, listeners FROM _staging_artists
    """)
    con.unregister("_staging_artists")
    saved = _db_count(con, "geo_top_artists", country)
    if saved != len(rows):
        msg = f"{country} artists: fetched {len(rows)} but db has {saved}"
        log.warning("  save mismatch: %s", msg)
        run_errors.append(f"save mismatch — {msg}")
    return False


def upsert_tracks(con, rows: list[dict], force: bool, run_errors: list) -> bool:
    if not rows:
        return True
    if not force and _is_unchanged(con, "geo_top_tracks", rows[0]["country"], rows):
        log.info("  tracks unchanged, skipping")
        return True
    country = rows[0]["country"]
    df = pd.DataFrame(rows)
    con.register("_staging_tracks", df)
    con.execute("DELETE FROM geo_top_tracks WHERE country = ?", [country])
    con.execute("""
        INSERT INTO geo_top_tracks (country, rank, track, track_url, artist, artist_mbid, artist_url, listeners, playcount)
        SELECT country, rank, track, track_url, artist, artist_mbid, artist_url, listeners, playcount FROM _staging_tracks
    """)
    con.unregister("_staging_tracks")
    saved = _db_count(con, "geo_top_tracks", country)
    if saved != len(rows):
        msg = f"{country} tracks: fetched {len(rows)} but db has {saved}"
        log.warning("  save mismatch: %s", msg)
        run_errors.append(f"save mismatch — {msg}")
    return False


def upsert_global_artists(con, rows: list[dict], run_errors: list) -> bool:
    if not rows:
        return True
    df = pd.DataFrame(rows)
    con.register("_staging_global_artists", df)
    con.execute("DELETE FROM global_top_artists")
    con.execute("""
        INSERT INTO global_top_artists (rank, artist, artist_mbid, artist_url, listeners, playcount)
        SELECT rank, artist, artist_mbid, artist_url, listeners, playcount FROM _staging_global_artists
    """)
    con.unregister("_staging_global_artists")
    saved = _db_count(con, "global_top_artists", None)
    if saved != len(rows):
        msg = f"global artists: fetched {len(rows)} but db has {saved}"
        log.warning("  save mismatch: %s", msg)
        run_errors.append(f"save mismatch — {msg}")
    return False


def upsert_global_tracks(con, rows: list[dict], run_errors: list) -> bool:
    if not rows:
        return True
    df = pd.DataFrame(rows)
    con.register("_staging_global_tracks", df)
    con.execute("DELETE FROM global_top_tracks")
    con.execute("""
        INSERT INTO global_top_tracks (rank, track, track_mbid, track_url, artist, artist_mbid, artist_url, listeners, playcount)
        SELECT rank, track, track_mbid, track_url, artist, artist_mbid, artist_url, listeners, playcount FROM _staging_global_tracks
    """)
    con.unregister("_staging_global_tracks")
    saved = _db_count(con, "global_top_tracks", None)
    if saved != len(rows):
        msg = f"global tracks: fetched {len(rows)} but db has {saved}"
        log.warning("  save mismatch: %s", msg)
        run_errors.append(f"save mismatch — {msg}")
    return False


def upsert_global_tags(con, rows: list[dict], run_errors: list) -> bool:
    if not rows:
        return True
    df = pd.DataFrame(rows)
    con.register("_staging_global_tags", df)
    con.execute("DELETE FROM global_top_tags")
    con.execute("""
        INSERT INTO global_top_tags (rank, tag, tag_url, reach, taggings)
        SELECT rank, tag, tag_url, reach, taggings FROM _staging_global_tags
    """)
    con.unregister("_staging_global_tags")
    saved = _db_count(con, "global_top_tags", None)
    if saved != len(rows):
        msg = f"global tags: fetched {len(rows)} but db has {saved}"
        log.warning("  save mismatch: %s", msg)
        run_errors.append(f"save mismatch — {msg}")
    return False

## Run

Executes the fetch using the configuration set at the top of the notebook.

In [ ]:
run_errors: list[str] = []
run_ts = datetime.now().strftime("%Y%m%d_%H%M%S")

only_countries = [c.strip() for c in ONLY_COUNTRIES if c.strip()]
if only_countries:
    unknown = [c for c in only_countries if c not in COUNTRY_CODES]
    assert not unknown, f"Unknown country names: {', '.join(unknown)}"

network = pylast.LastFMNetwork(
    api_key=os.environ["LASTFM_API_KEY"],
    api_secret=os.environ["LASTFM_API_SECRET"],
)

# Remove stale WAL left by a crashed previous session before connecting.
_wal = f"{DB_PATH}.wal"
if os.path.exists(_wal):
    os.remove(_wal)
    print(f"Removed stale WAL: {_wal}")

con = duckdb.connect(DB_PATH)
schema_changed = setup_db(con)

if schema_changed:
    log.info("Schema changed in tables: %s — forcing full refresh for those tables", schema_changed)


def _age(tbl: str, base_hours: float) -> float:
    """Return 0.0 (force refresh) if the table schema changed this run, else base_hours."""
    return 0.0 if tbl in schema_changed else base_hours


if not only_countries:
    log.info("--- Global charts (max-age %.0fh) ---", _age("global_top_artists", GLOBAL_MAX_AGE_HOURS))
    t0 = time.monotonic()
    if not upsert_global_artists(con, fetch_global_artists(network, con, run_errors, _age("global_top_artists", GLOBAL_MAX_AGE_HOURS)), run_errors):
        log.info("  saved global artists in %.2fs", time.monotonic() - t0)
    time.sleep(REQUEST_DELAY)
    t0 = time.monotonic()
    if not upsert_global_tracks(con, fetch_global_tracks(network, con, run_errors, _age("global_top_tracks", GLOBAL_MAX_AGE_HOURS)), run_errors):
        log.info("  saved global tracks in %.2fs", time.monotonic() - t0)
    time.sleep(REQUEST_DELAY)
    t0 = time.monotonic()
    if not upsert_global_tags(con, fetch_global_tags(network, con, run_errors, _age("global_top_tags", GLOBAL_MAX_AGE_HOURS)), run_errors):
        log.info("  saved global tags in %.2fs", time.monotonic() - t0)
    time.sleep(REQUEST_DELAY)

run_countries = only_countries if only_countries else ALL_COUNTRIES
log.info("--- Country charts (%d, max-age %.0fh) ---", len(run_countries), GEO_MAX_AGE_HOURS)
total = len(run_countries)
country_stats = []

for i, country in enumerate(run_countries, start=1):
    api_country = LASTFM_COUNTRY_NAME_MAP.get(country, country)
    if api_country is None:
        log.info("[%d/%d] %s — skipped (not supported by Last.fm)", i, total, country)
        country_stats.append({"country": country, "artists": 0, "tracks": 0, "status": "unsupported"})
        continue

    log.info("[%d/%d] %s", i, total, country)
    stat = {"country": country, "artists": 0, "tracks": 0, "status": "ok"}

    artists, force = fetch_geo_artists(network, api_country, con, run_errors, _age("geo_top_artists", GEO_MAX_AGE_HOURS))
    t0 = time.monotonic()
    skipped = upsert_artists(con, artists, force, run_errors)
    if not skipped:
        log.info("  saved artists in %.2fs", time.monotonic() - t0)
        stat["artists"] = len(artists)
    elif not artists:
        stat["status"] = "skipped" if not any(api_country in e for e in run_errors) else "failed"
    time.sleep(REQUEST_DELAY)

    tracks, force = fetch_geo_tracks(network, api_country, con, run_errors, _age("geo_top_tracks", GEO_MAX_AGE_HOURS))
    t0 = time.monotonic()
    skipped = upsert_tracks(con, tracks, force, run_errors)
    if not skipped:
        log.info("  saved tracks in %.2fs", time.monotonic() - t0)
        stat["tracks"] = len(tracks)
    time.sleep(REQUEST_DELAY)

    country_stats.append(stat)

con.close()
log.info("Done. DB saved to %s", DB_PATH)

## Summary

In [ ]:
con = duckdb.connect(DB_PATH, read_only=True)

geo_artists  = con.execute("SELECT COUNT(*) FROM geo_top_artists").fetchone()[0]
geo_tracks   = con.execute("SELECT COUNT(*) FROM geo_top_tracks").fetchone()[0]
glob_artists = con.execute("SELECT COUNT(*) FROM global_top_artists").fetchone()[0]
glob_tracks  = con.execute("SELECT COUNT(*) FROM global_top_tracks").fetchone()[0]
glob_tags    = con.execute("SELECT COUNT(*) FROM global_top_tags").fetchone()[0]
con.close()

summary = {
    "run_at": run_ts,
    "db": DB_PATH,
    "global_max_age_hours": GLOBAL_MAX_AGE_HOURS,
    "geo_max_age_hours": GEO_MAX_AGE_HOURS,
    "global": {"artists": glob_artists, "tracks": glob_tracks, "tags": glob_tags},
    "geo": {"artists": geo_artists, "tracks": geo_tracks},
    "errors": run_errors,
    "countries": country_stats,
}

summary_path = f"/kaggle/working/summary_{run_ts}.json"
Path(summary_path).write_text(json.dumps(summary, indent=2))

print(f"Global  — artists: {glob_artists:,}  tracks: {glob_tracks:,}  tags: {glob_tags:,}")
print(f"Geo     — artists: {geo_artists:,}  tracks: {geo_tracks:,}")
print(f"Errors  — {len(run_errors)}")
print(f"Summary — {summary_path}")
if run_errors:
    print("\nErrors:")
    for e in run_errors:
        print(" ", e)

## Export

Exports all four tables to **Parquet** and **CSV** under `/kaggle/working/`. The raw **DuckDB** database (`trends.db`) is also available — choose whichever format suits your workflow:

| Format | Path | Best for |
|---|---|---|
| DuckDB | `trends.db` | SQL queries, joins across tables, analytics |
| Parquet | `parquet/<table>.parquet` | pandas / polars / columnar analytics |
| CSV | `csv/<table>.csv` | spreadsheets, any tool |

**DuckDB:**
```python
import duckdb
con = duckdb.connect("trends.db", read_only=True)
con.execute("SELECT * FROM geo_top_artists WHERE country = 'Brazil' ORDER BY rank LIMIT 10").df()
```

**Parquet:**
```python
import pandas as pd
df = pd.read_parquet("parquet/geo_top_artists.parquet")
```

**CSV:**
```python
import pandas as pd
df = pd.read_csv("csv/geo_top_artists.csv")
```

In [ ]:
import os

TABLES = [
    "geo_top_artists",
    "geo_top_tracks",
    "global_top_artists",
    "global_top_tracks",
    "global_top_tags",
]

EXPORT_BASE = "/kaggle/working"
PARQUET_DIR = f"{EXPORT_BASE}/parquet"
CSV_DIR     = f"{EXPORT_BASE}/csv"

os.makedirs(PARQUET_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

con = duckdb.connect(DB_PATH, read_only=True)

for table in TABLES:
    parquet_path = f"{PARQUET_DIR}/{table}.parquet"
    csv_path     = f"{CSV_DIR}/{table}.csv"

    con.execute(f"COPY {table} TO '{parquet_path}' (FORMAT PARQUET)")
    con.execute(f"COPY {table} TO '{csv_path}' (FORMAT CSV, HEADER TRUE)")

    rows = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    pq_size  = os.path.getsize(parquet_path)
    csv_size = os.path.getsize(csv_path)
    print(f"{table}: {rows:,} rows | parquet {pq_size/1024:.0f} KB | csv {csv_size/1024:.0f} KB")

con.close()
print(f"\nFiles written to {EXPORT_BASE}/")
print("  parquet/  — columnar, compressed, fast for analytics")
print("  csv/      — plain text, compatible with any tool")

In [ ]:
readme_content = """# Last.fm Global Trends — Dataset

Top artists, tracks, and tags from the [Last.fm API](https://www.last.fm/api), covering every supported country plus global charts.

## Files

| File | Description |
|---|---|
| `trends.db` | DuckDB database (all tables) |
| `parquet/geo_top_artists.parquet` | Top artists per country (Parquet) |
| `parquet/geo_top_tracks.parquet` | Top tracks per country (Parquet) |
| `parquet/global_top_artists.parquet` | Global top artists (Parquet) |
| `parquet/global_top_tracks.parquet` | Global top tracks (Parquet) |
| `parquet/global_top_tags.parquet` | Global top tags (Parquet) |
| `csv/geo_top_artists.csv` | Top artists per country (CSV) |
| `csv/geo_top_tracks.csv` | Top tracks per country (CSV) |
| `csv/global_top_artists.csv` | Global top artists (CSV) |
| `csv/global_top_tracks.csv` | Global top tracks (CSV) |
| `csv/global_top_tags.csv` | Global top tags (CSV) |

## Tables & Columns

### `geo_top_artists` — Top artists per country

| Column | Type | Description |
|---|---|---|
| `country` | string | Country display name (e.g. `France`, `Türkiye`) |
| `rank` | int | Chart position (1 = most listeners) |
| `artist` | string | Artist name |
| `artist_mbid` | string | MusicBrainz ID for the artist (may be empty) |
| `artist_url` | string | Last.fm URL for the artist page |
| `listeners` | int | Listener count in that country |
| `fetched_at` | timestamp | When this row was last written to the DB |

### `geo_top_tracks` — Top tracks per country

| Column | Type | Description |
|---|---|---|
| `country` | string | Country display name |
| `rank` | int | Chart position (1 = most listeners) |
| `track` | string | Track title |
| `track_url` | string | Last.fm URL for the track page |
| `artist` | string | Artist name |
| `artist_mbid` | string | MusicBrainz ID for the artist (may be empty) |
| `artist_url` | string | Last.fm URL for the artist page |
| `listeners` | int | Listener count in that country |
| `playcount` | int | Play count in that country |
| `fetched_at` | timestamp | When this row was last written to the DB |

### `global_top_artists` — Global top artists

| Column | Type | Description |
|---|---|---|
| `rank` | int | Chart position (1 = most listeners globally) |
| `artist` | string | Artist name |
| `artist_mbid` | string | MusicBrainz ID for the artist (may be empty) |
| `artist_url` | string | Last.fm URL for the artist page |
| `listeners` | int | Total global listener count |
| `playcount` | int | Total global play count |
| `fetched_at` | timestamp | When this row was last written to the DB |

### `global_top_tracks` — Global top tracks

| Column | Type | Description |
|---|---|---|
| `rank` | int | Chart position (1 = most played globally) |
| `track` | string | Track title |
| `track_mbid` | string | MusicBrainz ID for the track (may be empty) |
| `track_url` | string | Last.fm URL for the track page |
| `artist` | string | Artist name |
| `artist_mbid` | string | MusicBrainz ID for the artist (may be empty) |
| `artist_url` | string | Last.fm URL for the artist page |
| `listeners` | int | Total global listener count |
| `playcount` | int | Total global play count |
| `fetched_at` | timestamp | When this row was last written to the DB |

### `global_top_tags` — Global top tags

Top 10,000 tags by usage. The Last.fm API reports ~2.8 M total tags; this table captures the most popular ones.

| Column | Type | Description |
|---|---|---|
| `rank` | int | Chart position (1 = most used) |
| `tag` | string | Tag name (e.g. `rock`, `electronic`, `seen live`) |
| `tag_url` | string | Last.fm URL for the tag page |
| `reach` | int | Number of distinct users who applied this tag |
| `taggings` | int | Total number of times this tag has been applied |
| `fetched_at` | timestamp | When this row was last written to the DB |

## Usage

### DuckDB

```python
import duckdb

con = duckdb.connect("trends.db", read_only=True)

# Top 10 artists in Brazil
con.execute(\"SELECT rank, artist, listeners, artist_url FROM geo_top_artists WHERE country = 'Brazil' ORDER BY rank LIMIT 10\").df()

# Countries where an artist appears in the top 50
con.execute(\"SELECT country, rank, listeners FROM geo_top_artists WHERE artist = 'Taylor Swift' AND rank <= 50 ORDER BY rank\").df()

# Global top 20 tracks with links
con.execute(\"SELECT rank, track, artist, playcount, track_url FROM global_top_tracks ORDER BY rank LIMIT 20\").df()

# Top 20 tags by reach
con.execute(\"SELECT rank, tag, reach, taggings FROM global_top_tags ORDER BY rank LIMIT 20\").df()

con.close()
```

### pandas

```python
import pandas as pd

df = pd.read_parquet(\"parquet/geo_top_artists.parquet\")
df = pd.read_csv(\"csv/global_top_tags.csv\")
```

## Update frequency

Data is refreshed weekly. Each run skips tables whose data is less than 7 days old and whose row count matches the API.
"""

readme_path = "/kaggle/working/README.md"
with open(readme_path, "w") as f:
    f.write(readme_content)

print(f"README written to {readme_path} ({len(readme_content):,} bytes)")